In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [2]:
GEN_1_GENETIC = CF_OUTPUTS / "gen1_models_explainers_constraints" / "genetic_exp"
GEN_1_GENETIC.is_dir()

True

In [3]:
batch_data = GEN_1_GENETIC / "XGB_highthres_2026-04-17" / "annotated.csv"
batch_df = pd.read_csv(batch_data)

### set PD options

In [4]:
pd.set_option("display.max_rows", None)

In [5]:
# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [6]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

In [7]:
# correct_cf_idx
# Fix cf_id indexing - group by query_index and number counterfactuals sequentially
def fix_cf_id(df):
    # For each query_index group
    fixed_rows = []
    for query_idx, group in df.groupby('query_index', sort=False):
        group = group.copy()
        # First row is xin (original)
        xin_mask = group['cf_id'] == 'xin'
        cf_mask = ~xin_mask

        # Keep xin rows as is
        xin_rows = group[xin_mask].copy()

        # Fix cf rows
        cf_rows = group[cf_mask].copy()
        if len(cf_rows) > 0:
            # Number them cf_1, cf_2, cf_3, etc.
            cf_rows['cf_id'] = [f'cf_{i+1}' for i in range(len(cf_rows))]

        # Combine back
        fixed_rows.append(xin_rows)
        if len(cf_rows) > 0:
            fixed_rows.append(cf_rows)

    return pd.concat(fixed_rows, ignore_index=True)

In [8]:
batch_df = fix_cf_id(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.22,,,,0.026,
1,0,cf_1,,,,,,,18.9866,,,0.0,1,True,0.026,0.0123
2,0,cf_2,,1,5,,1,,18.9866,,,0.2917,4,True,0.026,0.0026
3,0,cf_3,,,6,7,,,18.9745,,,0.2141,3,False,0.026,0.1428
4,0,cf_4,,,5,,,,19.0311,4,,0.2817,3,True,0.026,0.0081
5,0,cf_5,2,4,6,1,,,18.9866,,,0.2792,5,False,0.026,0.0485
6,0,cf_6,2,1,5,,1,,18.9866,,,0.3417,5,True,0.026,0.0037
7,0,cf_7,6,1,5,7,,,18.9866,,,0.2792,5,True,0.026,0.0017
8,0,cf_8,2,,6,5,,,18.9866,5,,0.3208,5,False,0.026,0.0799
9,0,cf_9,1,1,4,,1,,18.9866,,,0.325,5,False,0.026,0.018


In [9]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation
# prefers valid CFs without violations, and lowest Gower
single_gower_df = select_one_cf_per_query(batch_df, selector="gower")
single_gower_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.22,,,,0.026,
9,0,cf_1,,,,,,,18.9866,,,0.0,1,True,0.026,0.0123
1,1,xin,3,4,1,2,3,0,22.3757,0,1.3,,,,0.2497,
10,1,cf_2,,,,,1,,22.3757,,,0.0833,2,True,0.2497,0.07
2,2,xin,5,3,1,1,4,0,22.694,7,1.08,,,,0.218,
11,2,cf_7,,,2,2,1,,22.68,,,0.267,4,True,0.218,0.0124
3,3,xin,3,3,6,6,2,0,24.3418,1,1.18,,,,0.0653,
12,3,cf_1,,,,,,,24.3418,,,0.0001,1,False,0.0653,0.0653
4,4,xin,5,4,2,7,1,0,21.2585,3,1.31,,,,0.0811,
13,4,cf_1,,,,,,,21.2585,,,0.1118,1,True,0.0811,0.0141


In [10]:
# Random selection (Gower may pick only BMI variations)
single_random_df = select_one_cf_per_query(batch_df, selector="random")
single_random_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.22,,,,0.026,
9,0,cf_2,,1,5,,1,,18.9866,,,0.2917,4,True,0.026,0.0026
1,1,xin,3,4,1,2,3,0,22.3757,0,1.3,,,,0.2497,
10,1,cf_2,,,,,1,,22.3757,,,0.0833,2,True,0.2497,0.07
2,2,xin,5,3,1,1,4,0,22.694,7,1.08,,,,0.218,
11,2,cf_3,1,,,,2,,22.6757,,,0.3333,3,True,0.218,0.0662
3,3,xin,3,3,6,6,2,0,24.3418,1,1.18,,,,0.0653,
12,3,cf_1,,,,,,,24.3418,,,0.0001,1,False,0.0653,0.0653
4,4,xin,5,4,2,7,1,0,21.2585,3,1.31,,,,0.0811,
13,4,cf_2,,2,5,,,,21.2585,,,0.3618,3,True,0.0811,0.0091
